### Scraping the data and using LLM to ask and answer a question

In [5]:
#initialize environment variables
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
#Langchain Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true" #Enables langchain tracing
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT") #Check on the portal to track the traces of requests  

Load the file -> Split the file in chunks -> Convert each chunk to an vector embedding -> Store it in a vector database like Faiss/ChromaDB

In [6]:
#Building a scraper
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(web_path="https://www.langchain.com/blog/how-do-i-speed-up-my-agent")
docs = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [9]:
print(type(docs))
print(type(docs[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
documents = text_splitter.split_documents(docs)

In [11]:
documents

[Document(metadata={'source': 'https://www.langchain.com/blog/how-do-i-speed-up-my-agent', 'title': 'AI Agent Latency 101: How do I speed up my AI agent?', 'description': 'Learn proven strategies to speed up your AI agent: reduce latency, optimize LLM calls, enable parallelism, and improve UX. Expert tips from LangChain.', 'language': 'en'}, page_content='AI Agent Latency 101: How do I speed up my AI agent?\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLangChain has raised a $300M Series C at a $3B valuation|Learn more ↗\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nAgent ImprovementEngineImprove agents autonomouslyObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceAgent RuntimeAgent ServerShip and scale agents in productionSandboxesRun agent-generated code safelyFleetAgents for the whole companyAgent GovernanceLLM GatewayControl agent model callsOpen Source\n\nAgent FrameworksdeepagentsBuild long-

In [12]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [13]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [14]:
vectorstoredb

In [ ]:
# llm            = the model
# prompt         = template with {context}
# document_chain = prompt + llm        (needs "context" at run time)
# retriever      = FAISS search        (needs "input" at run time)
# retrieval_chain= retriever -> document_chain


# Correct. You pass {"input": "your question"} to retrieval_chain.
# Correct. The retrieval chain gives the question to the retriever. The retriever embeds it, searches FAISS, and returns the closest documents. The default is 4, and you can change it with as_retriever(search_kwargs={"k": ...}).
# Correct. The retrieval chain bundles the retriever’s output together with your original input into one dictionary, {"input": ..., "context": [docs]}, and passes it to the document chain.
# Mostly correct. The document chain joins the text of those documents into one string, puts it into {context} in your prompt, and sends the filled prompt to the LLM. The LLM writes the answer.

In [ ]:
## method 1 using similarity search which gives us the words letters in the proximity of the query
## Query from VectorDB
query = "Where the latency is coming from?"
result = vectorstoredb.similarity_search(query)
print(result[0])
print(result[0].page_content)

page_content='I get this question a bunch. Developers generally first spend time getting the agent to work, but then they turn their attention to speed and cost. There are few things we see developers doing:Identifying where the latency is coming fromChanging the UX to reduce the “perceived” latencyMaking fewer LLM callsSpeeding up LLM callsMaking LLM calls in parallelIdentifying where the latency is coming fromThis may sound basic, but how you approach reducing latency will depend entirely on your specific bottleneck. Is the latency coming from one big LLM call, or from multiple small ones that just add up? You’ll need to diagnose these before trying to speed it up.LangSmith is an incredibly useful tool for this, providing complete visibility into your agent interactions. You can track the latency of' metadata={'source': 'https://www.langchain.com/blog/how-do-i-speed-up-my-agent', 'title': 'AI Agent Latency 101: How do I speed up my AI agent?', 'description': 'Learn proven strategies 

In [41]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-6-luna")

In [ ]:
# Without {input}

# The LLM receives this:

# Answer the following question based only on the provided context:
# <context>
# ...retrieved chunks about latency...
# </context>

# There is no question in it. Depending on the model, you would typically get one of these:

# A general summary of the chunks, such as “The context says latency depends on your bottleneck and that LangSmith can help track it.”
# A reply asking what the question is.
# A guess at a question, answered from the context.

# With your latency question and latency documents, the summary can look like a decent answer by coincidence. The problem shows up when the question is narrow. If you asked “Which tool shows latency per step?” the model would still produce the same general summary, because it never saw that question.

# With {input}

# The LLM receives this:

# Answer the following question based only on the provided context:
# <context>
# ...retrieved chunks about latency...
# </context>

# Question: Where the latency is coming from?

# Now it answers that specific question from the chunks, for example: “It could come from one large LLM call or from many small calls adding up, and you need to diagnose which. LangSmith’s waterfall view can show which step contributes most.”


# Why it matters
# Focused answers. The model picks the relevant parts of the context instead of summarizing all of it.
# “I don’t know” works. Your prompt says “based only on the provided context.” If the chunks don’t contain the answer, the model can say so, but only if it knows what was asked.
# Different questions give different answers. Without {input}, the question influences the output only through which chunks were retrieved.

In [49]:
#Retrieval Chain and Document Chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
        {context}
    </context>


    Question: {input}
    """
)
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n        {context}\n    </context>\n\n\n    Question: {input}\n    '), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0', 'langchain-openai': '1.8.0'}}, profile={'name': 'GPT-6 Luna', 'release_date': '2026-09-22', 'last_updated': '2026-09-22', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': Fa

In [43]:
# Not needed just testing: Creating manual context here
# from langchain_core.documents import Document
# document_chain.invoke(
#     {
#         "context": [Document(page_content="This may sound basic, but how you approach reducing latency will depend entirely on your specific bottleneck. Is the latency coming from one big LLM call, or from multiple small ones that just add up? You'll need to diagnose these before trying to speed it up. LangSmith is an incredibly useful tool for this, providing complete visibility into your agent interactions. You can track the latency of each step of the agent, and we've also recently introduced a “waterfall” view to easily identify which steps contribute the most to overall latency.")]
#     }
# )

However, we want the documents to first come from the retriever we just set up. 
That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [44]:
### Input ---> Retriever ---> vectorstoredb
vectorstoredb

In [45]:
retriever = vectorstoredb.as_retriever() #as_retriever retrieves relevant documents
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [46]:
response = retrieval_chain.invoke({"input": "Where the latency is coming from?"})
response

{'input': 'Where the latency is coming from?',
 'context': [Document(id='db5929aa-2aca-4fe1-972d-ce378305da2e', metadata={'source': 'https://www.langchain.com/blog/how-do-i-speed-up-my-agent', 'title': 'AI Agent Latency 101: How do I speed up my AI agent?', 'description': 'Learn proven strategies to speed up your AI agent: reduce latency, optimize LLM calls, enable parallelism, and improve UX. Expert tips from LangChain.', 'language': 'en'}, page_content='I get this question a bunch. Developers generally first spend time getting the agent to work, but then they turn their attention to speed and cost. There are few things we see developers doing:Identifying where the latency is coming fromChanging the UX to reduce the “perceived” latencyMaking fewer LLM callsSpeeding up LLM callsMaking LLM calls in parallelIdentifying where the latency is coming fromThis may sound basic, but how you approach reducing latency will depend entirely on your specific bottleneck. Is the latency coming from on

In [33]:
response['answer']

'To speed up an AI agent:\n\n- **Find the bottleneck first.** Use tracing tools such as LangSmith to measure each step and identify whether latency comes from one large LLM call or many smaller calls.\n- **Reduce perceived latency.** Stream results so users can see the agent working while it runs.\n- **Reduce or optimize LLM calls.** Consider making fewer calls, speeding up calls, or running independent calls in parallel. Which approach helps depends on what your latency traces show.'

In [34]:
response['context']

[Document(id='db5929aa-2aca-4fe1-972d-ce378305da2e', metadata={'source': 'https://www.langchain.com/blog/how-do-i-speed-up-my-agent', 'title': 'AI Agent Latency 101: How do I speed up my AI agent?', 'description': 'Learn proven strategies to speed up your AI agent: reduce latency, optimize LLM calls, enable parallelism, and improve UX. Expert tips from LangChain.', 'language': 'en'}, page_content='I get this question a bunch. Developers generally first spend time getting the agent to work, but then they turn their attention to speed and cost. There are few things we see developers doing:Identifying where the latency is coming fromChanging the UX to reduce the “perceived” latencyMaking fewer LLM callsSpeeding up LLM callsMaking LLM calls in parallelIdentifying where the latency is coming fromThis may sound basic, but how you approach reducing latency will depend entirely on your specific bottleneck. Is the latency coming from one big LLM call, or from multiple small ones that just add u